# 07 · As a Tool — 04 when a tool fails

**Runs end to end with no API key.** Every failure below is produced by a deterministic scripted service — no randomness, no network, no sleeping. One optional cell hands a failure envelope to a real model if a key is loaded; without one it says so and shows the stand-in's recovery instead.

A tool that always works teaches nothing. The interesting part of tool use is the part where the call does not come back clean, and the single most important judgement in an agent is this one:

- **(a) Hand it back.** The failure is something the model can react to and try again around — a malformed argument, a transient timeout, an empty result. The right move is to turn it into a *result*: a normal, readable message the model receives and gets another turn on.
- **(b) Halt the run.** The failure is something no retry can fix — a missing credential, a revoked permission, an exhausted budget, the same error N times in a row. The right move is to stop, loudly.

Get this distinction wrong in the (a) direction — treat a fatal error as retryable — and the agent calls a broken tool forever, burning tokens on an error message that will never change. Get it wrong in the (b) direction — treat a recoverable error as fatal — and the agent gives up on questions it could have answered on the second try.

There is a third case, and it is the dangerous one: the tool that does not fail at all, and returns something confidently wrong. Nothing raises, so nothing catches it. Only a post-condition does. That is Step 10.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `ToolFailure` / `ToolTimeout` | Failures that go back to the model as a result it can act on. | `ToolFailure("days_between is missing required argument(s): end_date")` |
| `FatalToolError` | Failures that must stop the run. Never caught by `safe_call`. | `FatalToolError("SHIPPING_API_KEY is not set")` |
| `validate_arguments` | Rejects a bad call against the spec before the tool ever runs. | missing `end_date` -> an envelope, not a crash |
| `safe_call` | One call, with a retry budget, returning a result envelope either way. | `{"ok": False, "error": "...", "retryable": True}` |
| `agent_loop` | The loop, with and without the fatal/retryable distinction. | 12 wasted attempts vs. 1 and a halt |
| `check_result` | A post-condition that catches a plausible, schema-valid, wrong answer. | a delivery date in the year 2125 |

## Step 1 — bootstrap the repo path

In [ ]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()

## Step 2 — which path this run is on

Step 11 uses a real model if a key is loaded. Everything else is deterministic and offline.

In [ ]:
nbio.show_environment()

## Step 3 — two exception classes, and the line between them

The whole notebook rests on these six lines. `ToolFailure` means *this call* went wrong; `FatalToolError` means *this run* is over. The classification does not live in the loop or in the model — it lives at the point where the failure is raised, which is the only place with enough information to make it.

In [ ]:
class ToolFailure(Exception):
    """This call failed. The message goes back to the model, which gets another turn."""


class ToolTimeout(ToolFailure):
    """A transient failure. Worth retrying on its own before handing anything back."""


class FatalToolError(Exception):
    """No retry can fix this. The run stops here."""


print("ToolTimeout is a ToolFailure     :", issubclass(ToolTimeout, ToolFailure))
print("FatalToolError is a ToolFailure  :", issubclass(FatalToolError, ToolFailure))
assert not issubclass(FatalToolError, ToolFailure), "the fatal class must not be catchable as a normal failure"

## Step 4 — the tools, and a scripted service that fails on purpose

`days_between` is the same trivial tool from `03-a-second-tool.ipynb`. `shipment_status` calls a `ScriptedService`: a list of behaviours it works through one call at a time. No randomness — the same cell produces the same failures every run, which is the only way an assertion below can mean anything.

In [ ]:
from datetime import date


def days_between(start_date: str, end_date: str) -> int:
    """Count the number of whole calendar days between two dates, each written as YYYY-MM-DD.

    Args:
        start_date: The earlier date, as YYYY-MM-DD.
        end_date: The later date, as YYYY-MM-DD.
    """
    return (date.fromisoformat(end_date) - date.fromisoformat(start_date)).days


class ScriptedService:
    """A stand-in for a remote service. Each call consumes the next scripted behaviour."""

    def __init__(self, script: list):
        self.script = list(script)
        self.calls = 0

    def __call__(self, order_id: str) -> dict:
        self.calls += 1
        behaviour = self.script[min(self.calls - 1, len(self.script) - 1)]
        if behaviour == "timeout":
            raise ToolTimeout(f"shipment service did not respond in 5s for {order_id}")
        if behaviour == "no-key":
            raise FatalToolError("SHIPPING_API_KEY is not set -- every call will fail identically")
        if behaviour == "unknown-order":
            raise ToolFailure(f"no order {order_id!r}; order ids look like 'A-1042'")
        return dict(behaviour)  # a successful payload


SHIPMENT_SPEC = {
    "name": "shipment_status",
    "description": "Look up the shipping status and delivery date of one order.",
    "parameters": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "An order id such as A-1042."}},
        "required": ["order_id"],
    },
}
DAYS_SPEC = {
    "name": "days_between",
    "description": "Count the number of whole calendar days between two dates, each written as YYYY-MM-DD.",
    "parameters": {
        "type": "object",
        "properties": {
            "start_date": {"type": "string", "description": "The earlier date, as YYYY-MM-DD."},
            "end_date": {"type": "string", "description": "The later date, as YYYY-MM-DD."},
        },
        "required": ["start_date", "end_date"],
    },
}

delivered = {"order_id": "A-1042", "status": "delivered", "delivered_on": "2026-08-19"}
flaky = ScriptedService(["timeout", "timeout", delivered])
print(f"scripted behaviours queued for the flaky service: {len(flaky.script)}")

## Step 5 — the cheapest failure to handle: arguments that do not match the schema

A model that omits a required argument has made a mistake the spec already describes. Checking the arguments against the spec costs nothing, happens before the tool runs, and produces an error message that tells the model exactly what to send instead. This is failure class (a) in its purest form: the tool was never even called.

In [ ]:
def validate_arguments(spec: dict, arguments: dict) -> None:
    """Raise ToolFailure if the arguments do not satisfy the spec. Never fatal."""
    schema = spec["parameters"]
    missing = [k for k in schema["required"] if k not in arguments]
    if missing:
        raise ToolFailure(
            f"{spec['name']} is missing required argument(s): {', '.join(missing)}. "
            f"Required: {schema['required']}."
        )
    unknown = [k for k in arguments if k not in schema["properties"]]
    if unknown:
        raise ToolFailure(f"{spec['name']} got unknown argument(s): {', '.join(unknown)}.")


try:
    validate_arguments(DAYS_SPEC, {"start_date": "2026-08-02"})
except ToolFailure as exc:
    print(f"caught before the tool ran : {exc}")

validate_arguments(DAYS_SPEC, {"start_date": "2026-08-02", "end_date": "2026-09-04"})
print("a complete argument set validates cleanly")

## Step 6 — `safe_call`: one call, a retry budget, and an envelope either way

`safe_call` is the whole (a)/(b) decision in one function. Note what it does **not** catch: `FatalToolError` is re-raised on purpose, before any retry logic can see it. A bare `except Exception` in this position is the single most common way an agent ends up looping on a broken tool — it converts "this will never work" into "this did not work this time".

The backoff schedule is recorded rather than slept through, so the notebook stays fast. A real client would sleep those seconds.

In [ ]:
def safe_call(spec: dict, fn, arguments: dict, *, attempts: int = 3) -> dict:
    """Call a tool and always return an envelope -- unless the failure is fatal, which propagates."""
    backoff: list[int] = []
    for attempt in range(1, attempts + 1):
        try:
            validate_arguments(spec, arguments)
            return {"ok": True, "tool": spec["name"], "value": fn(**arguments),
                    "attempts": attempt, "backoff_s": backoff}
        except FatalToolError:
            raise  # (b) -- no retry can fix this, and the loop must not get a chance to try
        except ToolTimeout as exc:
            if attempt < attempts:
                backoff.append(2 ** (attempt - 1))
                continue
            message = f"{exc} (gave up after {attempt} attempts)"
        except ToolFailure as exc:
            message = str(exc)
        except Exception as exc:
            message = f"{type(exc).__name__}: {exc}"
        return {"ok": False, "tool": spec["name"], "error": message, "retryable": True,
                "attempts": attempt, "backoff_s": backoff}


result = safe_call(DAYS_SPEC, days_between, {"start_date": "2026-08-02"})
nbio.show_json(result)
assert result["ok"] is False and result["retryable"] is True
assert "end_date" in result["error"], "the message has to name what to fix"

## Step 7 — (a) in full: a bad call, an error the model can read, a corrected retry

Three envelopes, in the order an agent would actually produce them. The second one is the point of the whole class: a date in the wrong format raises deep inside `date.fromisoformat`, and what comes back to the caller is not a traceback but a one-line message. The third is the corrected call succeeding.

Compare the two failure messages in the output, because they are not equally good. The first was written here and names the fix (`missing required argument(s): end_date. Required: [...]`). The second is whatever the standard library happened to say (`Invalid isoformat string: '08/02/2026'`) — it names the problem and not the remedy, and a model has to infer `YYYY-MM-DD` from the argument description instead of being told. Error messages are prompts: rewriting that second one at the wrapper is real work with a real payoff, and this notebook deliberately leaves it raw so the difference is visible.

In [ ]:
import json

attempts = [
    {"start_date": "2026-08-02"},                              # missing an argument
    {"start_date": "08/02/2026", "end_date": "2026-09-04"},    # right shape, wrong format
    {"start_date": "2026-08-02", "end_date": "2026-09-04"},    # corrected
]

rows = []
for arguments in attempts:
    envelope = safe_call(DAYS_SPEC, days_between, arguments)
    outcome = str(envelope["value"]) if envelope["ok"] else envelope["error"]
    rows.append((json.dumps(arguments)[:52], "ok" if envelope["ok"] else "failed", outcome[:62]))

nbio.table(rows, headers=("arguments", "", "value or message"))

final = safe_call(DAYS_SPEC, days_between, attempts[-1])
assert final["ok"] and final["value"] == 33
print()
print("Every one of those three calls returned. None of them raised into the loop.")

## Step 8 — a transient failure, retried inside the call

The scripted service times out twice and succeeds on the third call. The model never sees the first two — retrying a timeout is the tool wrapper's job, not a decision worth a model turn. What the envelope records is that it took three attempts and what the backoff schedule would have been.

The retry budget is what makes this bounded. Without it, "retry a timeout" and "loop forever" are the same code.

In [ ]:
envelope = safe_call(SHIPMENT_SPEC, flaky, {"order_id": "A-1042"})
nbio.show_json(envelope)

assert envelope["ok"] is True, "the third scripted attempt succeeds"
assert envelope["attempts"] == 3
assert envelope["backoff_s"] == [1, 2], "two failures, two backoff waits"
assert flaky.calls == 3, "the service really was called three times"
print()
print(f"the model sees one result; the wrapper absorbed {flaky.calls - 1} timeouts")

## Step 9 — (b) in full: the failure that must stop the run

A missing credential produces the identical error on every call, forever. `safe_call` does not catch it, does not retry it, and does not turn it into a message the model will cheerfully try to work around. It propagates, and the run ends.

The cell below then shows the cost of getting this wrong, by running the same broken tool through a loop that treats every failure as retryable — the exact loop-on-a-broken-tool failure this distinction exists to prevent. It is capped at 12 attempts so the cell terminates; nothing inside it would ever have stopped on its own.

In [ ]:
broken = ScriptedService(["no-key"])

try:
    safe_call(SHIPMENT_SPEC, broken, {"order_id": "A-1042"})
    raised = None
except FatalToolError as exc:
    raised = exc
print(f"safe_call propagated : {type(raised).__name__}: {raised}")
assert isinstance(raised, FatalToolError), "a fatal error must not be swallowed into an envelope"
assert broken.calls == 1, "and it must not be retried even once"


def agent_loop(spec, fn, arguments, *, honour_fatal: bool, max_turns: int = 12) -> dict:
    """Call a tool until it works. With honour_fatal=False, 'until it works' can mean never."""
    errors = []
    for turn in range(1, max_turns + 1):
        try:
            envelope = safe_call(spec, fn, arguments, attempts=1)
        except FatalToolError as exc:
            if honour_fatal:
                return {"turns": turn, "halted": True, "reason": str(exc)}
            errors.append(str(exc))
            continue  # treat it as just another bad turn, and go round again
        if envelope["ok"]:
            return {"turns": turn, "halted": False, "reason": "succeeded"}
        errors.append(envelope["error"])
    return {"turns": max_turns, "halted": False, "reason": f"hit the {max_turns}-turn cap, still failing"}


naive = agent_loop(SHIPMENT_SPEC, ScriptedService(["no-key"]), {"order_id": "A-1042"}, honour_fatal=False)
correct = agent_loop(SHIPMENT_SPEC, ScriptedService(["no-key"]), {"order_id": "A-1042"}, honour_fatal=True)

nbio.table(
    [("every failure is retryable", naive["turns"], naive["reason"][:52]),
     ("fatal errors halt", correct["turns"], correct["reason"][:52])],
    headers=("loop policy", "turns spent", "outcome"),
)

assert naive["turns"] == 12 and not naive["halted"], "the naive loop only stopped because of the cap"
assert correct["halted"] and correct["turns"] == 1, "the correct loop stops on the first fatal error"
print()
print("12 turns against an error that could not change, versus 1. In a real loop each of")
print("those turns is a model call with the full conversation attached to it.")

## Step 10 — the third case: the tool that succeeds and is wrong

Nothing raises here. `shipment_status` returns a well-formed payload that matches its schema, and one field in it is nonsense — a delivery date in the year 2125, the kind of thing an upstream typo or a unit mix-up produces. Every mechanism built above is blind to it, because every mechanism built above is triggered by an exception.

A post-condition is the only thing that catches this: an explicit check of whether the *value* is possible, written by whoever knows what possible means. The check turns a wrong answer into failure class (a) — a message the model can react to — instead of a fact it quietly repeats to a user.

In [ ]:
from datetime import date as _date

corrupt = {"order_id": "A-1042", "status": "delivered", "delivered_on": "2125-08-19"}
service = ScriptedService([corrupt])

raw = safe_call(SHIPMENT_SPEC, service, {"order_id": "A-1042"})
print("the call succeeded, and the envelope says so:")
nbio.show_json(raw)


def check_result(envelope: dict) -> dict:
    """A post-condition on a successful call. Turns an impossible value into a readable failure."""
    if not envelope.get("ok"):
        return envelope
    value = envelope["value"]
    delivered_on = value.get("delivered_on") if isinstance(value, dict) else None
    if delivered_on:
        parsed = _date.fromisoformat(delivered_on)
        if not (_date(2000, 1, 1) <= parsed <= _date(2030, 12, 31)):
            return {"ok": False, "tool": envelope["tool"], "retryable": True, "attempts": envelope["attempts"],
                    "error": f"delivered_on={delivered_on!r} is outside the plausible range 2000-2030; "
                             f"the upstream record is probably wrong. Do not report this date."}
    return envelope


checked = check_result(raw)
print()
print("and the post-condition says otherwise:")
nbio.show_json(checked)

assert raw["ok"] is True, "no exception was raised -- that is exactly the problem"
assert checked["ok"] is False and "plausible range" in checked["error"]
assert check_result(safe_call(SHIPMENT_SPEC, ScriptedService([delivered]), {"order_id": "A-1042"}))["ok"] is True
print()
print("Same tool, same schema, same envelope shape. One value is possible and one is not,")
print("and only a check written by someone who knows the difference can tell them apart.")

## Step 11 — the four failures, classified

Every failure this notebook produced, with the class it was given and the reason. The reason column is the part that has to be written by a human for each tool: nothing can infer from a stack trace whether a retry could ever help.

In [ ]:
CLASSIFICATION = [
    ("missing argument", "hand back (a)", "the spec already says what to send instead"),
    ("bad date format", "hand back (a)", "a corrected call succeeds; the message says how"),
    ("timeout", "retry, then hand back (a)", "a later attempt can differ; bounded by a budget"),
    ("missing credential", "halt (b)", "every attempt returns the identical error, forever"),
    ("impossible value", "hand back (a), via a post-condition", "nothing raised, so nothing else could catch it"),
]
nbio.table(CLASSIFICATION, headers=("failure", "class", "why"))

assert sum(1 for _, cls, _ in CLASSIFICATION if cls.startswith("halt")) == 1
print()
print("One of five halts. That ratio is the point: halting is rare and specific, and")
print("everything else is a turn the model gets to take.")

## Step 12 — handing a failure envelope to a real model — only if a key is loaded

The claim behind failure class (a) is that a model reads the error and fixes its call. With a key loaded, this cell tests that claim directly: it sends the bad-format call, hands back the real error message as a tool result, and prints what the model asks for next — under a spend ceiling. Without a key it says so and prints the stand-in's recovery instead, which is the corrected call written by hand in Step 7.

In [ ]:
import os


def _client():
    if os.environ.get("GROQ_API_KEY"):
        from groq import Groq

        return "groq", Groq(api_key=os.environ["GROQ_API_KEY"]), "llama-3.3-70b-versatile"
    if os.environ.get("OPENAI_API_KEY"):
        from openai import OpenAI

        return "openai", OpenAI(api_key=os.environ["OPENAI_API_KEY"]), "gpt-4o-mini"
    return None, None, None


bad_call_envelope = safe_call(DAYS_SPEC, days_between, {"start_date": "08/02/2026", "end_date": "2026-09-04"})
provider, client, model_id = _client()

with nbio.cost_meter(budget_usd=0.50) as meter:
    if provider is None:
        print(
            "No GROQ_API_KEY or OPENAI_API_KEY loaded -- not set, skipping. Running the "
            "deterministic stand-in instead: the recovery shown here is the corrected call "
            "written by hand in Step 7, not a model's decision."
        )
        print(f"  error handed back : {bad_call_envelope['error']}")
        print(f"  stand-in retry    : days_between('2026-08-02', '2026-09-04') -> "
              f"{days_between('2026-08-02', '2026-09-04')}")
    else:
        print(f"provider={provider!r} model={model_id!r}")
        messages = [
            {"role": "user", "content": "My order shipped 08/02/2026. How many days until 2026-09-04?"},
            {"role": "assistant", "content": None, "tool_calls": [{
                "id": "call_1", "type": "function",
                "function": {"name": "days_between",
                             "arguments": '{"start_date": "08/02/2026", "end_date": "2026-09-04"}'},
            }]},
            {"role": "tool", "tool_call_id": "call_1", "content": bad_call_envelope["error"]},
        ]
        resp = client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=[{"type": "function", "function": DAYS_SPEC}],
            temperature=0.0,
        )
        usage = getattr(resp, "usage", None)
        meter.record(model_id,
                     getattr(usage, "prompt_tokens", 0) if usage else 0,
                     getattr(usage, "completion_tokens", 0) if usage else 0)
        retry = resp.choices[0].message.tool_calls or []
        print(f"  error handed back : {bad_call_envelope['error']}")
        print(f"  model's next move : {retry[0].function.arguments if retry else resp.choices[0].message.content}")

print()
print(meter.report())

## What did not come across

- **Nothing here actually timed out.** `ScriptedService` raises `ToolTimeout` instantly and the backoff schedule is recorded rather than slept through. A real client sets a socket timeout, and the hardest part of real timeout handling — a request that is still running on the other end after you gave up on it — is entirely absent.
- **Retrying is only free for reads.** Every tool in this notebook is a lookup. Retrying a tool that *writes* something can double-charge a card or send two emails, and the fix for that is idempotency keys, which are not here.
- **No circuit breaker, and no memory between runs.** A tool that failed fatally in this run is fresh again in the next one. Real systems stop calling a tool that has been failing for everyone for the last ten minutes.
- **`FatalToolError` has to be raised by someone.** The classification lives at the raise site, and this notebook's raise sites are hand-written to demonstrate it. Wrapping a real client library means reading its exception hierarchy and deciding, error by error, which side of the line each one falls on. Nothing automates that.
- **Spend is capped elsewhere.** The runaway this notebook shows is a loop; the runaway that costs money is metered by `nbio.cost_meter`, and rate limits are `04-retrieve/07-rate-limiting.ipynb`. Three different ceilings, three different things being capped.
- **Nothing is recorded.** Every envelope above is printed and dropped. Which tool failed, how often, and how long it took is stage `05-observe`.

This is the last notebook in `07-as-a-tool`. What comes next is the rest of the agent: `02-memory` (what it remembers between turns), `03-guardrails` (what it is not allowed to do), `04-orchestrate` (more than one choice, in sequence), and `05-observe` (what actually happened).